# Project Foundations for Data Science: FoodHub Data Analysis**Marks: 60 points***Author: Samuel Odudimu — MIT-PE AAIDSP (May'26, Batch D)*

### ContextThe number of restaurants in New York is increasing day by day. Lots of students and busy professionals rely on those restaurants due to their hectic lifestyles. A food aggregator company **FoodHub** offers access to multiple restaurants through a single smartphone app. The app lets restaurants receive direct online orders; the company assigns a delivery person to pick up and deliver each order, and the customer can rate the order. FoodHub earns a fixed margin on each delivery order.### ObjectiveFoodHub has stored data on orders placed by registered customers. The Data Science team wants to analyze this data to understand the demand across restaurants and improve the customer experience. As a Data Scientist, we perform an exploratory data analysis (EDA) to answer the key business questions below.### Data Dictionary- **order_id**: Unique ID of the order- **customer_id**: ID of the customer who ordered the food- **restaurant_name**: Name of the restaurant- **cuisine_type**: Cuisine ordered by the customer- **cost_of_the_order**: Cost of the order- **day_of_the_week**: Whether the order was placed on a Weekday or Weekend- **rating**: Rating given by the customer out of 5 (`Not given` if unrated)- **food_preparation_time**: Minutes taken by the restaurant to prepare the food- **delivery_time**: Minutes taken by the delivery person to deliver the food

### Let us start by importing the required libraries

In [ ]:
import numpy as npimport pandas as pdimport matplotlib.pyplot as pltimport seaborn as snssns.set_theme(style='whitegrid')plt.rcParams['figure.figsize'] = (8, 5)pd.set_option('display.float_format', lambda x: f'{x:.2f}')

### Loading the dataIn Google Colab, run the cell below and upload `foodhub_order.csv` when prompted. (If the file is already in the working directory, it loads directly.)

In [ ]:
import osif not os.path.exists('foodhub_order.csv'):    try:        from google.colab import files        uploaded = files.upload()    except Exception:        passdf = pd.read_csv('foodhub_order.csv')df.head()

### Understanding the structure of the data

### **Question 1:** How many rows and columns are present in the data? [0.5 mark]

In [ ]:
df.shape

**Observation:** The dataset has **1,898 rows** (orders) and **9 columns** (features).

### **Question 2:** What are the datatypes of the different columns in the dataset? [0.5 mark]

In [ ]:
df.info()

**Observation:** There are **9 columns**:- 4 integer columns: `order_id`, `customer_id`, `food_preparation_time`, `delivery_time`- 1 float column: `cost_of_the_order`- 4 object (text) columns: `restaurant_name`, `cuisine_type`, `day_of_the_week`, `rating`Note that `rating` is stored as text because it contains the label `Not given` for unrated orders alongside the numeric ratings.

### **Question 3:** Are there any missing values in the data? If yes, treat them using an appropriate method. [1 Mark]

In [ ]:
df.isnull().sum()

**Observation:** There are **no missing (null) values** in any column, so no imputation is required. The `rating` column does, however, encode unrated orders with the text value `Not given` rather than a true null — we account for this explicitly wherever numeric ratings are needed (Questions 6 and 13).

### **Question 4:** Check the statistical summary of the data. What is the minimum, average, and maximum time it takes for food to be prepared once an order is placed? [2 marks]

In [ ]:
df.describe()

In [ ]:
print('Food preparation time (minutes)')print('Minimum :', df['food_preparation_time'].min())print('Average :', round(df['food_preparation_time'].mean(), 2))print('Maximum :', df['food_preparation_time'].max())

**Observation:** Food preparation time ranges from a **minimum of 20 minutes** to a **maximum of 35 minutes**, with an **average of ~27.37 minutes**. The spread is narrow (15 minutes), suggesting restaurants are fairly consistent in preparation time.

### **Question 5:** How many orders are not rated? [1 mark]

In [ ]:
df['rating'].value_counts()

**Observation:** **736 orders** are unrated (`Not given`) — about **38.8%** of all orders. This is a substantial share and limits how much we can rely on ratings for quality decisions; encouraging more customers to rate would improve feedback coverage.

### Exploratory Data Analysis (EDA)

### Univariate Analysis

### **Question 6:** Explore all the variables and provide observations on their distributions. (Generally, histograms, boxplots, countplots, etc. are used for univariate exploration.) [9 marks]

#### Order ID and Customer ID

In [ ]:
print('Unique order IDs    :', df['order_id'].nunique(), 'of', len(df))print('Unique customer IDs :', df['customer_id'].nunique(), 'of', len(df))

**Observation:** Every `order_id` is unique (1,898 distinct values), confirming one row per order. There are **1,200 unique customers**, so many customers placed more than one order — there is scope for repeat-customer and loyalty analysis.

#### Restaurant name

In [ ]:
print('Number of unique restaurants:', df['restaurant_name'].nunique())plt.figure(figsize=(9,6))df['restaurant_name'].value_counts().head(10).plot(kind='barh')plt.title('Top 10 Restaurants by Number of Orders')plt.xlabel('Number of orders')plt.gca().invert_yaxis()plt.show()

**Observation:** Orders are spread across **178 restaurants**, but demand is highly concentrated: a small group (**Shake Shack, The Meatball Shop, Blue Ribbon Sushi, Blue Ribbon Fried Chicken, Parm**) accounts for a disproportionate share of orders, while the long tail of restaurants receives only a handful each.

#### Cuisine type

In [ ]:
plt.figure(figsize=(9,5))sns.countplot(data=df, y='cuisine_type', order=df['cuisine_type'].value_counts().index)plt.title('Order Count by Cuisine Type')plt.xlabel('Number of orders')plt.show()

**Observation:** **American** is the most ordered cuisine, followed by **Japanese** and **Italian**; together these three dominate demand. Cuisines such as Vietnamese, Spanish, Korean and French are ordered rarely.

#### Cost of the order

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(12,4))sns.histplot(df['cost_of_the_order'], bins=30, kde=True, ax=ax[0])ax[0].set_title('Distribution of Order Cost')sns.boxplot(x=df['cost_of_the_order'], ax=ax[1])ax[1].set_title('Boxplot of Order Cost')plt.show()

**Observation:** Order cost ranges from **$4.47 to $35.41**, with a **mean of ~$16.50** and a **median of ~$14.14**. The distribution is mildly right-skewed (mean > median) with no extreme outliers — most orders fall between roughly $12 and $23.

#### Day of the week

In [ ]:
plt.figure(figsize=(5,4))sns.countplot(data=df, x='day_of_the_week')plt.title('Orders by Day Type')plt.show()print(df['day_of_the_week'].value_counts())

**Observation:** The majority of orders — **1,351 (≈71%)** — are placed on **weekends**, versus **547 on weekdays**. Demand is strongly weekend-driven, which matters for staffing and delivery-fleet planning.

#### Rating

In [ ]:
plt.figure(figsize=(5,4))sns.countplot(data=df, x='rating', order=['Not given','3','4','5'])plt.title('Distribution of Ratings')plt.show()

**Observation:** The most common value is **`Not given` (736 orders)**. Among rated orders, **5 is the most frequent (588)**, then 4 (386) and 3 (188); there are **no 1- or 2-star ratings**. Rated experiences skew positive, but the large unrated share means satisfaction is only partially observed.

#### Food preparation time and Delivery time

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(12,4))sns.histplot(df['food_preparation_time'], bins=15, kde=True, ax=ax[0])ax[0].set_title('Food Preparation Time')sns.histplot(df['delivery_time'], bins=15, kde=True, ax=ax[1])ax[1].set_title('Delivery Time')plt.show()

**Observation:** **Food preparation time** is roughly uniform between **20 and 35 minutes** (mean ≈ 27.4). **Delivery time** ranges from **15 to 33 minutes** (mean ≈ 24.2) and is somewhat left-skewed, with many deliveries clustered toward the higher end. Neither variable shows extreme outliers.

### **Question 7:** Which are the top 5 restaurants in terms of the number of orders received? [1 mark]

In [ ]:
df['restaurant_name'].value_counts().head(5)

**Observation:** The top 5 restaurants by order volume are **Shake Shack (219), The Meatball Shop (132), Blue Ribbon Sushi (119), Blue Ribbon Fried Chicken (96), and Parm (68)**. These five alone account for a large portion of total orders.

### **Question 8:** Which is the most popular cuisine on weekends? [1 mark]

In [ ]:
weekend = df[df['day_of_the_week'] == 'Weekend']weekend['cuisine_type'].value_counts().head()

**Observation:** On weekends, **American** is the most popular cuisine (415 orders), ahead of Japanese (335) and Italian (207).

### **Question 9:** What percentage of the orders cost more than 20 dollars? [2 marks]

In [ ]:
above_20 = df[df['cost_of_the_order'] > 20]pct = 100 * len(above_20) / len(df)print('Orders above $20:', len(above_20))print(f'Percentage: {pct:.2f}%')

**Observation:** **555 orders (≈29.24%)** cost more than $20. Roughly three in ten orders are high-value, which is relevant for the company's commission revenue.

### **Question 10:** What is the mean order delivery time? [1 mark]

In [ ]:
round(df['delivery_time'].mean(), 2)

**Observation:** The **mean delivery time is ~24.16 minutes**.

### **Question 11:** The company has decided to give 20% discount vouchers to the top 5 most frequent customers. Find the IDs of these customers and the number of orders they placed. [1 mark]

In [ ]:
df['customer_id'].value_counts().head(5)

**Observation:** The top 5 most frequent customers are IDs **52832 (13 orders), 47440 (10), 83287 (9), 250494 (8), and 82041 (7)** — these should receive the 20% discount vouchers.

### Multivariate Analysis

### **Question 12:** Perform a multivariate analysis to explore relationships between the important variables in the dataset. [10 marks]

#### Cuisine vs Cost of the order

In [ ]:
plt.figure(figsize=(12,5))sns.boxplot(data=df, x='cuisine_type', y='cost_of_the_order')plt.xticks(rotation=45, ha='right')plt.title('Cost of Order by Cuisine Type')plt.show()

**Observation:** Median order cost is broadly similar across cuisines, but some cuisines (e.g., Korean, Spanish, Thai) show wider cost spreads while the high-volume cuisines (American, Japanese, Italian) have tight, comparable cost distributions.

#### Cuisine vs Food Preparation time

In [ ]:
plt.figure(figsize=(12,5))sns.boxplot(data=df, x='cuisine_type', y='food_preparation_time')plt.xticks(rotation=45, ha='right')plt.title('Food Preparation Time by Cuisine Type')plt.show()

**Observation:** Food preparation time is fairly uniform across cuisines (all centered around 25-30 minutes), indicating cuisine type is not a strong driver of how long food takes to prepare.

#### Day of the Week vs Delivery time

In [ ]:
plt.figure(figsize=(6,5))sns.boxplot(data=df, x='day_of_the_week', y='delivery_time')plt.title('Delivery Time: Weekday vs Weekend')plt.show()

**Observation:** **Weekday deliveries take noticeably longer** (median/mean ≈ 28 min) than **weekend deliveries** (≈ 22 min) — most likely due to weekday traffic congestion. This is examined numerically in Question 16.

#### Revenue generated by the restaurants

In [ ]:
plt.figure(figsize=(9,6))df.groupby('restaurant_name')['cost_of_the_order'].sum().sort_values(ascending=False).head(10).plot(kind='barh')plt.title('Top 10 Restaurants by Total Order Value')plt.xlabel('Total order value ($)')plt.gca().invert_yaxis()plt.show()

**Observation:** The restaurants generating the most revenue mirror the most-ordered restaurants — **Shake Shack, The Meatball Shop, Blue Ribbon Sushi and Blue Ribbon Fried Chicken** lead — confirming that a small set of partners drives the bulk of FoodHub's gross order value.

#### Rating vs Delivery time, Food preparation time, and Cost

In [ ]:
rated = df[df['rating'] != 'Not given'].copy()rated['rating'] = rated['rating'].astype(int)fig, ax = plt.subplots(1, 3, figsize=(15,4))sns.boxplot(data=rated, x='rating', y='delivery_time', ax=ax[0]); ax[0].set_title('Rating vs Delivery time')sns.boxplot(data=rated, x='rating', y='food_preparation_time', ax=ax[1]); ax[1].set_title('Rating vs Prep time')sns.boxplot(data=rated, x='rating', y='cost_of_the_order', ax=ax[2]); ax[2].set_title('Rating vs Cost')plt.show()

**Observation:** Among rated orders, delivery time, preparation time and cost are all fairly similar across the 3/4/5 rating levels — none shows a strong monotonic relationship with the rating. Customer ratings in this dataset are not clearly explained by these operational variables alone.

#### Correlation among numerical variables

In [ ]:
plt.figure(figsize=(6,5))sns.heatmap(df[['cost_of_the_order','food_preparation_time','delivery_time']].corr(), annot=True, cmap='coolwarm', fmt='.2f')plt.title('Correlation Heatmap')plt.show()

**Observation:** Correlations between `cost_of_the_order`, `food_preparation_time` and `delivery_time` are all very weak (near zero). These operational variables are essentially independent of one another.

### **Question 13:** The company wants to provide a promotional offer in the advertisement of the restaurants. The condition to get the offer is that the restaurants must have a rating count of more than 50 and the average rating should be greater than 4. Find the restaurants fulfilling the criteria to get the promotional offer. [3 marks]

In [ ]:
rated = df[df['rating'] != 'Not given'].copy()rated['rating'] = rated['rating'].astype(int)rating_count = rated.groupby('restaurant_name')['rating'].count()rating_mean = rated.groupby('restaurant_name')['rating'].mean()qualified = rating_count[rating_count > 50].indexpromo = rating_mean[qualified][rating_mean[qualified] > 4]promo.sort_values(ascending=False)

**Observation:** Four restaurants qualify for the promotional offer (rating count > 50 **and** average rating > 4): **The Meatball Shop (avg 4.51), Blue Ribbon Fried Chicken (4.33), Shake Shack (4.28), and Blue Ribbon Sushi (4.22)**. These are both popular and well-rated — ideal candidates to feature in advertising.

### **Question 14:** The company charges the restaurant 25% on the orders having cost greater than 20 dollars and 15% on the orders having cost greater than 5 dollars. Find the net revenue generated by the company across all orders. [3 marks]

In [ ]:
def revenue(cost):    if cost > 20:        return 0.25 * cost    elif cost > 5:        return 0.15 * cost    else:        return 0.0df['revenue'] = df['cost_of_the_order'].apply(revenue)print('Net revenue generated: $', round(df['revenue'].sum(), 2))

**Observation:** The company's **net revenue across all orders is ~$6,166.30**. High-value orders (>$20), charged at 25%, contribute disproportionately to this total.

### **Question 15:** The company wants to analyze the total time required to deliver the food. What percentage of orders take more than 60 minutes to get delivered from the time the order is placed? (The food has to be prepared and then delivered.) [2 marks]

In [ ]:
df['total_time'] = df['food_preparation_time'] + df['delivery_time']over_60 = df[df['total_time'] > 60]pct = 100 * len(over_60) / len(df)print('Orders over 60 minutes:', len(over_60))print(f'Percentage: {pct:.2f}%')

**Observation:** **200 orders (≈10.54%)** take more than 60 minutes end-to-end (preparation + delivery). About one in ten orders breaches the one-hour mark — a meaningful customer-experience risk.

### **Question 16:** The company wants to analyze the delivery time of the orders on weekdays and weekends. How does the mean delivery time vary during weekdays and weekends? [2 marks]

In [ ]:
df.groupby('day_of_the_week')['delivery_time'].mean().round(2)

**Observation:** Mean delivery time is **~28.34 minutes on weekdays** versus **~22.47 minutes on weekends** — deliveries are roughly **6 minutes faster on weekends**, most plausibly because of lighter weekend road traffic. More delivery staff on weekdays could close this gap.

### Conclusion and Recommendations

### **Question 17:** What are your conclusions from the analysis? What recommendations would you like to share to help improve the business? [6 marks]

### Conclusions:* **Demand is concentrated and weekend-heavy.** Just five restaurants (led by Shake Shack) and three cuisines (American, Japanese, Italian) drive most orders, and ~71% of all orders fall on weekends.* **Operations are consistent.** Food preparation (20-35 min) and delivery (15-33 min) times are stable with no extreme outliers, and cost, prep time and delivery time are essentially uncorrelated.* **Weekday deliveries are slower** (~28.3 min) than weekend deliveries (~22.5 min), and ~10.5% of all orders exceed 60 minutes end-to-end.* **Feedback coverage is weak.** Nearly 39% of orders are unrated; among rated orders sentiment is positive (no 1- or 2-star ratings), but satisfaction is only partially observed.* **A small set of restaurants is both popular and highly rated** (The Meatball Shop, Blue Ribbon Fried Chicken, Shake Shack, Blue Ribbon Sushi), and high-value orders (>$20, ~29%) drive most of the ~$6,166 net revenue.

### Recommendations:* **Incentivize ratings.** With ~39% of orders unrated, offer small loyalty points or discount nudges for leaving a rating to build a more reliable quality signal.* **Promote the proven winners.** Feature the four promo-eligible restaurants (rating count > 50 and average > 4) in advertising, and deepen partnerships with the top-cuisine, top-revenue restaurants to protect the core of the business.* **Fix weekday delivery speed.** Add delivery capacity (or dynamic routing) on weekdays to cut the ~6-minute delivery-time gap and reduce the share of >60-minute orders.* **Grow the long tail and under-served cuisines.** Spotlight smaller restaurants and low-volume cuisines (Thai, Korean, Mexican, etc.) through targeted recommendations to diversify demand beyond the few dominant partners.* **Reward loyal customers.** Extend the 20% voucher beyond the top 5 frequent customers into a tiered loyalty program to increase repeat orders, since many customers already order multiple times.* **Capture more high-value orders.** Because >$20 orders generate the most commission, use combo deals and free-delivery thresholds to nudge average order value upward.